# CNN

Este es el notebook desde el cual vamos a tomar las imagenes ya preprocesadas y se va a entrenar un modelo usando Resnet.


In [ ]:
# prompt: import resnet necessary libraries

import pandas as pd
from google.colab import drive
import time
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Input


In [ ]:
# prompt: read a csv in the same folder  in google drive named as dataframe.csv

drive.mount('/content/drive')

# Specify the path to your CSV file in Google Drive
file_path = '/content/drive/Shareddrives/ClasifEye/datos.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)
print(df.columns)
df.head(100)


In [ ]:
# prompt: delete all rows with mtcnn_confidence column bellow 85 or nan, n then delete that column

# Remove rows where 'mtcnn_confidence' is less than 85 or is NaN
df = df[(df['mtcnn_confidence'] <= 85) | (df['mtcnn_confidence'].isna())]
df.count()

# Remove the 'mtcnn_confidence' column
df = df.drop(columns=['mtcnn_confidence'])




# Normalizacion de datos

In [ ]:
# prompt: normalizate these columns: nose_x	nose_y	mouth_right_x	mouth_right_y	right_eye_x	right_eye_y	left_eye_x	left_eye_y	mouth_left_x	mouth_left_y
# using 224 as the max

max_value = 224
columns_to_normalize = ['nose_x', 'nose_y', 'mouth_right_x', 'mouth_right_y', 'right_eye_x', 'right_eye_y', 'left_eye_x', 'left_eye_y', 'mouth_left_x', 'mouth_left_y']

for col in columns_to_normalize:
    df[col] = df[col] / max_value

print(df[columns_to_normalize].head())

In [ ]:
# prompt: normalizate age using minmaxscaler

from sklearn.preprocessing import MinMaxScaler

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler to the 'age' column and transform it
df['edad'] = scaler.fit_transform(df[['edad']])

print(df.head())


In [ ]:
# prompt: encode sexo and raza columns using onehotencoder

from sklearn.preprocessing import OneHotEncoder

# Initialize OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform the 'sexo' and 'raza' columns
encoded_features = onehot_encoder.fit_transform(df[['sexo', 'raza']])

# Create a DataFrame with the encoded features
encoded_df = pd.DataFrame(encoded_features, columns=onehot_encoder.get_feature_names_out(['sexo', 'raza']))

# Concatenate the original DataFrame (excluding original 'sexo' and 'raza') with the encoded DataFrame
df = pd.concat([df.drop(['sexo', 'raza'], axis=1), encoded_df], axis=1)

print(df.head())

In [ ]:
# prompt: drop all rows with nan values in any column

df = df.dropna()
df.info()
df.describe()

# Creacion del modelo

In [ ]:
IMG_SIZE = 224  # Tamaño de imagen requerido por ResNet50
BATCH_SIZE = 32 # Ajusta según la memoria de tu GPU
EPOCHS = 10
image_dir = '/content/drive/Shareddrives/ClasifEye/Dataset Preprocesado'
df['archivo'] = df['archivo'].apply(lambda name: os.path.join(image_dir, name))

nombre_faltante = "50_0_3_20170113184235991.jpg"

# Elimina las filas que contengan ese archivo en la columna 'output_path'
df = df[~df['archivo'].str.contains(nombre_faltante)].reset_index(drop=True)

print("✅ Fila eliminada. Nuevo tamaño del DataFrame:", len(df))

In [ ]:

# ============================================================================
# COLUMNAS DE SALIDA
# ============================================================================
y_cols_edad = ['edad']
y_cols_sexo = ['sexo_Hombre', 'sexo_Mujer']
y_cols_raza = ['raza_Asiatico', 'raza_Blanco', 'raza_Indio', 'raza_Negro', 'raza_Otros']
all_y_cols = y_cols_edad + y_cols_sexo + y_cols_raza

# ============================================================================
# COLUMNAS EXTRA (coordenadas faciales)
# ============================================================================
extra_cols = [
    'nose_x', 'nose_y',
    'mouth_right_x', 'mouth_right_y',
    'right_eye_x', 'right_eye_y',
    'left_eye_x', 'left_eye_y',
    'mouth_left_x', 'mouth_left_y'
]

# ============================================================================
# CARGA Y LIMPIEZA DEL DATAFRAME
# ============================================================================
df_cleaned = df.dropna(subset=all_y_cols + extra_cols)

# ============================================================================
# FUNCIÓN DE PREPROCESAMIENTO
# ============================================================================
def load_and_preprocess_image(filepath, labels, extra_vars):
    image_string = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image_string, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = preprocess_input(image)

    label_edad = tf.cast(labels[0], tf.float32)
    label_sexo = tf.cast(labels[1:3], tf.float32)
    label_raza = tf.cast(labels[3:8], tf.float32)

    extra_vars = tf.cast(extra_vars, tf.float32)

    return (image, extra_vars), {
        'edad_output': label_edad,
        'sexo_output': label_sexo,
        'raza_output': label_raza
    }

# ============================================================================
# CREACIÓN DEL DATASET tf.data
# ============================================================================
dataset = tf.data.Dataset.from_tensor_slices((
    df_cleaned['archivo'].values,
    df_cleaned[all_y_cols].values.astype('float32'),
    df_cleaned[extra_cols].values.astype('float32')
))

dataset = (
    dataset.shuffle(buffer_size=len(df_cleaned))
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
# ============================================================================
# CONSTRUCCIÓN DEL MODELO CON ENTRADA EXTRA
# ============================================================================
def build_separate_heads_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), extra_dim=10):
    # Entrada imagen
    image_input = Input(shape=input_shape, name='image_input')

    # Entrada extra (coordenadas faciales)
    extra_input = Input(shape=(extra_dim,), name='extra_input')

    # Modelo base ResNet50 (pre-entrenado)
    base_model = ResNet50(include_top=False, weights='imagenet', input_tensor=image_input)
    base_model.trainable = False

    # Vector de características
    x = layers.GlobalAveragePooling2D()(base_model.output)

    # Concatenar con variables extra
    x = layers.Concatenate()([x, extra_input])

    # Capa densa compartida
    x = layers.Dense(512, activation='relu', name='shared_dense')(x)
    x = layers.Dropout(0.5)(x)

    # Rama edad
    age_branch = layers.Dense(128, activation='relu')(x)
    age_output = layers.Dense(1, activation='linear', name='edad_output')(age_branch)

    # Rama sexo
    gender_branch = layers.Dense(128, activation='relu')(x)
    gender_output = layers.Dense(len(y_cols_sexo), activation='softmax', name='sexo_output')(gender_branch)

    # Rama raza
    race_branch = layers.Dense(128, activation='relu')(x)
    race_output = layers.Dense(len(y_cols_raza), activation='softmax', name='raza_output')(race_branch)

    # Modelo final
    model = models.Model(
        inputs=[image_input, extra_input],
        outputs=[age_output, gender_output, race_output]
    )

    return model
# ============================================================================
# COMPILAR Y MOSTRAR EL MODELO
# ============================================================================
model = build_separate_heads_model()
model.compile(
    optimizer='adam',
    loss={
        'edad_output': 'mse',
        'sexo_output': 'categorical_crossentropy',
        'raza_output': 'categorical_crossentropy'
    },
    metrics={
        'edad_output': 'mae',
        'sexo_output': 'accuracy',
        'raza_output': 'accuracy'
    }
)

model.summary()

# Entrenamiento

In [ ]:
!ls "/content/drive/Shareddrives/ClasifEye/Dataset Preprocesado/50_0_3_20170113184235991.jpg"


In [ ]:
# Compilar el modelo especificando una pérdida y métrica para cada salida
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'edad_output': 'mean_squared_error',         # Pérdida para regresión
        'sexo_output': 'categorical_crossentropy',   # Pérdida para clasificación
        'raza_output': 'categorical_crossentropy'    # Pérdida para clasificación
    },
    metrics={
        'edad_output': 'mean_absolute_error', # Métrica interpretable (ej. "error de 3.5 años")
        'sexo_output': 'accuracy',
        'raza_output': 'accuracy'
    }
)

# Entrenar el modelo
print("\n--- INICIANDO ENTRENAMIENTO ---")
history = model.fit(
    dataset,
    epochs=EPOCHS
    # Si tuvieras un set de validación 'val_dataset', lo añadirías aquí:
    # validation_data=val_dataset
)
print("\n--- ENTRENAMIENTO FINALIZADO ---")



In [ ]:
# Entrenar el modelo
print("\n--- INICIANDO ENTRENAMIENTO ---")
history = model.fit(
    dataset,
    epochs=25,
    initial_epoch=10
)
print("\n--- ENTRENAMIENTO FINALIZADO ---")

In [ ]:
# Guardar el modelo entrenado
model.save('modelo_caras_resnet50.h5')